In [114]:
%load_ext dotenv
%dotenv /home/aurora/.env
%matplotlib inline

import numpy as np
import pandas as pd
import datetime as dt
from google.cloud import bigquery
import re
import pymongo
from bson import ObjectId
import os
import datetime as dt
from flatten_json import flatten
import seaborn as sns
import matplotlib.pyplot as plt
from pprint import pprint
import ipywidgets as widgets

client = bigquery.Client.from_service_account_json('/home/analytics/.secure_files/hitwicketsuperstars-f3e8c620a88c.json')

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [116]:
start = str(dt.date(2019,6,7))
end = str(dt.date(2019,6,20))
print(start,end)

2019-06-07 2019-06-20


In [117]:
app_version = '2.0.0'

In [118]:
query = (f"""
             SELECT 
             user_id as device_id,
             event_timestamp
             FROM `hitwicketsuperstars.analytics_190927423.events_*`,
             UNNEST(event_params) AS params    
             WHERE _TABLE_SUFFIX BETWEEN "{start.replace('-','')}"
             AND "{end.replace('-','')}"
             AND app_info.version = '{app_version}'
            AND device.operating_system = 'ANDROID'
             AND params.key = 'action'
            AND event_name IN ('natasha')
           AND params.value.string_value = 'hand_pointer_achievements_clicked'""")
         
ftue_complete = client.query(query).to_dataframe() 

In [119]:
ftue_complete.columns = ['device_id','create_time']
ftue_complete.sort_values('create_time',inplace=True,ascending=False)
ftue_complete.drop_duplicates('device_id',inplace=True)
print(len(ftue_complete))
ftue_complete.head()

849


,device_id,create_time
434,d6cc813cfb706d884fd08c9e4522d22d,1561045891117007
435,ec77e123a3a631702486098ecc3cf354,1561021170233006
433,2110e38a7b4fcec0b4f5427a40972572,1561020435836008
871,0491d97e7af60354f4b86a22dd486d9e,1560947808337007
788,318c1e527f12a89ff3d6bc4de60be163,1560938141314007


In [120]:
query = (f"""
             SELECT user_id as device_id,
                    event_timestamp,
                    params.value.string_value,
                    app_info.version
             FROM `hitwicketsuperstars.analytics_190927423.events_*`,
             UNNEST(event_params) AS params    
             WHERE _TABLE_SUFFIX BETWEEN "{start.replace('-','')}"
             AND "{end.replace('-','')}"
             AND device.operating_system = 'ANDROID'
            AND params.key = 'action'
             AND app_info.version = '{app_version}'
           
         """)
df = client.query(query).to_dataframe()

In [124]:
print(len(df))
df.head()

473886


,device_id,event_timestamp,string_value,version
0,1b9a131591952a0cf0ed75c8462b9c35,1560957923683001,natasha_team_shaping_well_seen,2.0.0
1,1b9a131591952a0cf0ed75c8462b9c35,1560957929197003,natasha_team_shaping_well_clicked,2.0.0
2,1b9a131591952a0cf0ed75c8462b9c35,1560957929208004,stadium_build_seen,2.0.0
3,b7ade8396eba0b2119a350df87172b22,1560965007400001,natasha_learning_quickly_seen,2.0.0
4,b7ade8396eba0b2119a350df87172b22,1560965007953001,natasha_stadium_ready_inaugrate_seen,2.0.0


In [125]:
new_users = pd.merge(ftue_complete,df,on='device_id')

new_users = new_users[new_users['event_timestamp']>new_users['create_time']] #ftue completed users
new_users = new_users[(new_users['event_timestamp']-new_users['create_time'])< (86400000000)] #activity in the first 24 hours

print(len(new_users))
new_users.head()

162697


,device_id,create_time,event_timestamp,string_value,version
92,d6cc813cfb706d884fd08c9e4522d22d,1561045891117007,1561045894851008,go,2.0.0
93,d6cc813cfb706d884fd08c9e4522d22d,1561045891117007,1561045896726002,store,2.0.0
94,d6cc813cfb706d884fd08c9e4522d22d,1561045891117007,1561045901361006,world_tour_clicked,2.0.0
95,d6cc813cfb706d884fd08c9e4522d22d,1561045891117007,1561045907896011,play,2.0.0
96,d6cc813cfb706d884fd08c9e4522d22d,1561045891117007,1561045912225014,toss_started,2.0.0


In [126]:
danu = new_users.device_id.nunique()
danu

839

# Training Center

## Visited Training Center

In [127]:
training_center_df = new_users[(new_users['string_value'] == 'training_center_clicked')]

In [128]:
len(new_users['device_id'].unique())

839

In [129]:
t = new_users[~new_users['device_id'].isin(new_users[new_users['string_value'] == 'training_center_clicked']['device_id'].unique())]
t = ((t.drop_duplicates('device_id'))['device_id']).reset_index()
print(len(t))
t.head()

176


,index,device_id
0,7118,0491d97e7af60354f4b86a22dd486d9e
1,8368,b97cc6225fc388b829c42b31905a5973
2,8705,f7dd687880bbd9560c411f76b25ee03c
3,8919,aa3ed0546f5e99b13df1d520b382d05f
4,9332,415d2ed10652ca1239bbbda42e3c04fc


In [131]:
t = pd.merge(t,ftue_complete,on='device_id')
t.head()

,index,device_id,create_time
0,7118,0491d97e7af60354f4b86a22dd486d9e,1560947808337007
1,8368,b97cc6225fc388b829c42b31905a5973,1560936441342007
2,8705,f7dd687880bbd9560c411f76b25ee03c,1560906630280007
3,8919,aa3ed0546f5e99b13df1d520b382d05f,1560877391748007
4,9332,415d2ed10652ca1239bbbda42e3c04fc,1560797074579022


In [132]:
t.to_csv('users_not_training_on_d0_old.csv')

In [29]:
training_page_interaction = (new_users[(new_users['string_value'] == 'training_center_clicked')]['device_id']).nunique()
training_page_interaction

792

In [30]:
training_page_interaction/danu*100 # percentage of people interacting with the training centre

70.77747989276139

## Put a player for Training

In [31]:
training_users_df = new_users[new_users['device_id'].isin(training_center_df['device_id'])]

In [32]:
new_users[new_users['string_value'].isin(['sent_to_training','sent_to_training_using_hitcoins'])]['device_id'].nunique() 

586

In [20]:
training_page_under_training = (new_users[new_users['string_value'].isin(['sent_to_training','sent_to_training_using_hitcoins'])]['device_id'].nunique() / training_page_interaction)*100

In [21]:
# Percentage of DAU
training_page_under_training

73.98989898989899

In [ ]:
training_page_under_training_visited = (len(df[df.string_value=='training_page_under_training']['user_id'].unique())/len(df[df.string_value=='training_page_idle']['user_id'].unique()))*100

In [ ]:
# Percentage of People who visited Training Center
training_page_under_training_visited

## Viewed Speedup Page

In [ ]:
speedup_panel_under_training = (len(df[df.string_value=='speedup_panel']['user_id'].unique())/len(df[df.string_value=='training_page_under_training']['user_id'].unique()))*100

In [ ]:
# Percentage of People who put Player for Training
speedup_panel_under_training

## Completed Training

In [ ]:
training_page_completed = (len(df[df.string_value=='training_page_completed']['user_id'].unique()) / dau)*100

In [ ]:
# Percentage of DAU
training_page_completed

In [ ]:
training_page_completed_visited = (len(df[df.string_value=='training_page_completed']['user_id'].unique())/len(df[df.string_value=='training_page_idle']['user_id'].unique()))*100

In [ ]:
# Percentage of People who visited Training Center
training_page_completed_visited

# Plotting

In [ ]:
import plotly.graph_objs as go
from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot
import plotly.io as pio
init_notebook_mode(connected=True)

In [ ]:
data = [go.Bar(
            x=['Daily Deals', 'Training Center', 'League Page', 'Mail Inbox', 'City Page', 'Scout Page', 'World Tour', 'Daily Task', 'Achievments'],
            y=[daily_deals, training_page_idle, league, mail, city, scout, world_tour, daily_task, achievements],
    text="%",
    marker=dict(
        color='rgb(158,202,225)',
        line=dict(
            color='rgb(8,48,107)',
            width=1.5,
        )
    ),
    opacity=0.6
    )]
plt = iplot(data, filename='% of DAU')

In [ ]:
import matplotlib.pyplot as plt

In [ ]:

height = [daily_deals, training_page_idle, league, mail, city, scout, world_tour, daily_task, achievements]
bars = ['Daily Deals', 'Training Center', 'League Page', 'Mail Inbox', 'City Page', 'Scout Page', 'World Tour', 'Daily Task', 'Achievments']
y_pos = np.arange(len(bars))
 
plt.figure(figsize=(20,10))
# Create bars and choose color
plt.bar(y_pos, height)
for i in range(len(bars)):
    plt.text(x = y_pos[i]-0.25, y = height[i] + 1, s = str(round(height[i],2))+"%", size = 15)

# Add title and axis names
plt.title('Visit % per page per DAU')
# plt.xlabel('Page Name', size=15)
plt.ylabel('Visit %', size=20)
 
# Limits for the Y axis
plt.ylim(0,100)
 
# Create names
plt.xticks(y_pos, bars, size=15)
 
plt.savefig(f'images/{date}.png')
# Show graphic
plt.show()
